VIDEO TO FRAMES

In [1]:
import cv2
import os

video_path = "sample_video2.mp4"
output_folder = "frames"

os.makedirs(output_folder, exist_ok=True)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

metadata_list = []
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    if frame_count % 12 == 0 and frame_count != 0:
        filename = f"frame_{frame_count:06d}.jpg"
        filepath = f"{output_folder}/{filename}"
        
        cv2.imwrite(filepath, frame)
        
        timestamp = frame_count / fps
        
        metadata_list.append({
            "frame_index": frame_count,
            "timestamp": round(timestamp, 2),
            "image_path": filepath
        })
        
    frame_count += 1

cap.release()
print(f"Đã trích xuất {len(metadata_list)} frames vào thư mục {output_folder}")
print("Metadata mẫu của frame đầu tiên:", metadata_list[0] if metadata_list else "Không có data")

Đã trích xuất 1432 frames vào thư mục frames
Metadata mẫu của frame đầu tiên: {'frame_index': 12, 'timestamp': 0.5, 'image_path': 'frames/frame_000012.jpg'}


FRAME TO VECTOR

In [2]:
import torch
import faiss
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

print("Downloading CLIP...")
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)

image_embeddings = []

print("Bắt đầu mã hóa (encode) các frame thành vector...")
for meta in metadata_list:
    image = Image.open(meta["image_path"])
    
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        features = model.get_image_features(**inputs)
        
        if not isinstance(features, torch.Tensor):
            features = features.pooler_output if hasattr(features, 'pooler_output') else features[0]
            
    embedding = features.cpu().numpy()
    
    faiss.normalize_L2(embedding)
    image_embeddings.append(embedding[0])

image_embeddings = np.array(image_embeddings).astype('float32')
print(f"Kích thước ma trận embedding: {image_embeddings.shape}")

dimension = image_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension) 

index.add(image_embeddings)
print(f"Đã thêm {index.ntotal} vector vào FAISS index.")

c:\Users\Hi\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 26460.81it/s]


Bắt đầu mã hóa (encode) các frame thành vector...
Kích thước ma trận embedding: (1432, 512)
Đã thêm 1432 vector vào FAISS index.


QUERY -> VECTOR -> SEARCH -> RESULT

In [3]:
query_text = "a hand holding a small computer chip" 
print(f"Top 10 results for: {query_text}")

text_inputs = processor(text=[query_text], return_tensors="pt", padding=True)
with torch.no_grad():
    text_features = model.get_text_features(**text_inputs)
    
    if not isinstance(text_features, torch.Tensor):
        text_features = text_features.pooler_output if hasattr(text_features, 'pooler_output') else text_features[0]
        
text_embedding = text_features.cpu().numpy()
faiss.normalize_L2(text_embedding)

k = 10
scores, indices = index.search(text_embedding, k)

for i in range(k):
    match_idx = indices[0][i]
    
    meta = metadata_list[match_idx]
    
    rank = i + 1
    frame = meta['frame_index']
    time = meta['timestamp']
    score = scores[0][i]
    path = meta['image_path']
    
    print(f"{rank}. frame={frame} time={time:.2f}s score={score:.3f} path={path}")

Top 10 results for: a hand holding a small computer chip
1. frame=9444 time=393.89s score=0.286 path=frames/frame_009444.jpg
2. frame=5736 time=239.24s score=0.282 path=frames/frame_005736.jpg
3. frame=5760 time=240.24s score=0.282 path=frames/frame_005760.jpg
4. frame=5724 time=238.74s score=0.281 path=frames/frame_005724.jpg
5. frame=15792 time=658.66s score=0.278 path=frames/frame_015792.jpg
6. frame=5772 time=240.74s score=0.278 path=frames/frame_005772.jpg
7. frame=15684 time=654.15s score=0.278 path=frames/frame_015684.jpg
8. frame=1272 time=53.05s score=0.277 path=frames/frame_001272.jpg
9. frame=10416 time=434.43s score=0.276 path=frames/frame_010416.jpg
10. frame=5748 time=239.74s score=0.275 path=frames/frame_005748.jpg
